# Notebook 00: Descarga de datos

Este notebook descarga los datos del taller desde Yahoo Finance y los guarda en disco para que el resto de notebooks los carguen rápidamente sin tener que volver a descargar.

**Solo hace falta ejecutarlo una vez al inicio del proyecto** (o si quieres actualizar los datos a fechas más recientes).

## Qué hace este notebook

1. Descarga el cierre ajustado de los 23 tickers del SP500 desde 1945.
2. Calcula los retornos logarítmicos diarios.
3. Guarda los datos en formato Parquet en `data/`.
4. Visualiza los retornos para verificar que todo está correcto.


## 1. Configuración del path

Antes de importar nada de `src/`, añadimos la raíz del proyecto al path de Python. Esto es necesario porque los notebooks se ejecutan desde la carpeta `notebooks/` pero los módulos están en `../src/`.

In [ ]:
import sys
from pathlib import Path

# Añadir la raíz del proyecto al PYTHONPATH para poder importar src/
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Raíz del proyecto: {project_root}")
print(f"¿Existe la carpeta src/? {(project_root / 'src').exists()}")

## 2. Imports

Importamos las funciones que necesitamos. Solo hacen falta:
- `descargar_datos` del módulo `src.data`.
- `plot_returns` del módulo `src.plotting` para visualizar.

In [ ]:
import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)

from src.data import descargar_datos, cargar_returns, cargar_precios, TICKERS, START_DATE
from src.plotting import plot_returns

print(f"Tickers configurados ({len(TICKERS)}): {TICKERS}")
print(f"Fecha de inicio: {START_DATE}")

## 3. Descarga (o carga desde caché)

La función `descargar_datos` automáticamente:
- Si NO existe el caché en `data/returns.parquet`, descarga de Yahoo Finance.
- Si ya existe, lo carga de disco (mucho más rápido).

Si quieres forzar una redescarga, pasa `force=True`.

In [ ]:
returns = descargar_datos(force=False, verbose=True)
print()
print(f"Forma de los retornos: {returns.shape}")
print(f"Rango de fechas: {returns.index.min().date()} a {returns.index.max().date()}")
returns.head()

## 4. Verificación rápida

Comprobamos que no hay valores extraños y que el shape es el esperado.

In [ ]:
# Estadísticas básicas por activo
returns.describe().T

In [ ]:
# ¿Hay valores nulos?
nulls = returns.isnull().sum()
if nulls.sum() == 0:
    print("OK: No hay valores nulos en los retornos.")
else:
    print("ATENCIÓN: hay valores nulos:")
    print(nulls[nulls > 0])

## 5. Visualización

Plotamos los retornos de los 23 activos para ver que la serie temporal tiene sentido (alta volatilidad concentrada en las crisis: 2000, 2008, 2020).

In [ ]:
fig, ax = plot_returns(
    returns,
    title=f"Retornos logarítmicos diarios ({returns.shape[1]} activos del SP500)",
    save_path=project_root / "results" / "figures" / "00_returns_completos.png",
)

## 6. Verificación: cargar desde caché

Comprobamos que la función `cargar_returns` funciona, que es la que usaremos en el resto de notebooks (sin volver a descargar).

In [ ]:
returns_cached = cargar_returns(verbose=True)
print(f"Forma: {returns_cached.shape}")
print(f"¿Es idéntico a lo descargado? {returns.equals(returns_cached)}")

## 7. Comprobar precios también

Por si en algún notebook necesitamos los precios crudos (no los retornos).

In [ ]:
precios = cargar_precios(verbose=True)
precios.head()

## Resumen

Si todas las celdas anteriores se ejecutaron sin error, ya tienes los datos guardados en:

- `data/returns.parquet`: retornos logarítmicos diarios.
- `data/precios_close.parquet`: precios de cierre ajustados.
- `results/figures/00_returns_completos.png`: gráfica de los retornos.

**Siguiente paso**: ejecutar el notebook `01_baselines.ipynb` para entrenar los baselines no neuronales (regresión lineal, persistencia, etc.).